# Fill in missing values based on municipalities

In [1]:
import pandas as pd
import geopandas as gpd

In [2]:
original = pd.read_csv('original_skolky_with_fee_v2.csv', index_col=0)
len(original)

5464

In [3]:
no_fee = original[original['monthly_fee'].isna()]
len(no_fee)

3750

In [4]:
adresses = gpd.read_file('skolky_points/skolky_points.shp')
obce = gpd.read_file('obce.shp', encoding='cp1250')
mc = gpd.read_file('m_casti.shp', encoding='cp1250')
# merge the two GeoDataFrames on 'Název ORP'

In [5]:
obce = obce[['NAZEV','NAZORP','geometry']]
obce = obce.rename(columns={'NAZEV': 'obec'})

In [6]:
mc = mc[['NAZEV', 'SPROBVOD','geometry']]
mc = mc.rename(columns={'NAZEV': 'mc', 'SPROBVOD': 'spravni_obvod'})

In [7]:
join  = adresses.sjoin(obce)
join =join.drop(columns=['index_right'])
join2 = join.sjoin(mc, how='left')
join2['IČO'] = join2['IČO'].astype(int)

In [8]:
join_original = pd.merge(original, join2[['obec', 'mc', 'spravni_obvod', 'NAZORP', 'IČO']], on='IČO', how='left')

In [9]:
obce_dict = {
    "Most": 700,
    "Litvínov": 570,
    "Teplice": 650,
    "Pardubice": 830,
    "Kroměříž": 600,
    "Louny": 950,
    "Duchcov": 500,
    "Brno-Židenice": 790,
    "Brno-Královo pole": 1000,
    "Brno-Líšeň": 800,
    "Třebíč": 800,
    "Praha 3": 1300,
    "Praha 5": 1200,
    "Praha 6": 1200,
    "Praha 7": 1500,
    "Praha 8": 1500,
    "Praha 9": 1350,
    "Kladno": 850,
    "Mladá Boleslav": 500,
    "Příbram": 600,
    "České Budějovice": 420,
    "Písek": 450,
    "Plzeň": 700,
    "Uherský Brod": 800,
    "Opava": 470,
    "Ostrov": 600,
    "Chomutov": 600,
    "Stodůlky": 900,
    "Brno-sever": 1000,
    "Ostrava-jih": 600,
    "Poruba": 750,
    "Velké Meziříčí": 400,
    "Zlín": 600,
    "Praha 4": 1300,
    "Praha-Čakovice": 1350,
    "Slezská Ostrava": 1000,
    "Moravská Ostrava a Přívoz": 1000,
    "Přerov": 600,
    "Valašské Mezíříčí": 800,
    "Vsetín": 500,
    "Tábor": 500
}


In [10]:
# Update the 'monthly_fee' if it is NaN
join_original['monthly_fee'] = join_original.apply(
    lambda row: obce_dict.get(row['obec'], row['monthly_fee']) if pd.isna(row['monthly_fee']) else row['monthly_fee'],
    axis=1
)

In [11]:
no_fee = join_original[join_original['monthly_fee'].isna()]
len(no_fee)

3492

In [12]:
join_original['monthly_fee'] = join_original.apply(
        lambda row: obce_dict.get(row['mc'], row['monthly_fee']) if pd.isna(row['monthly_fee']) else row['monthly_fee'],
        axis=1
 )


In [13]:
no_fee = join_original[join_original['monthly_fee'].isna()]
len(no_fee)

3354

In [14]:
join_original.columns

Index(['IČO', 'Zřizovatel', 'Území', 'Kraj', 'Okres/Obvod', 'Správní úřad',
       'ORP', 'Název ORP', 'Plný název', 'Zkrácený název', 'Ulice', 'Č. p.',
       'Č. or.', 'Část obce', 'PSČ', 'Místo', 'WWW', 'ZUJ', 'monthly_fee',
       'meals', 'obec', 'mc', 'spravni_obvod', 'NAZORP'],
      dtype='object')

In [15]:
join_original.to_csv('interoplated_original_skolky.csv', index=False)